# Raw Episode Collection

Collect raw Super Mario Bros. level 1-1 episodes (aiming for around 80 episodes for initial smoke-test)

Artifact structure:

```text
data/raw/episodes/ep_000001/
  frames.mkv     # lossless video, preferably FFV1
  steps.jsonl    # one JSON row per frame/action
  meta.json      # episode-level metadata
```

In [29]:
# Initialize the data collection directory structure
from __future__ import annotations

import sys
import time
import gzip
import json
import random
import shutil
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any

import numpy as np

try:
    import imageio.v2 as imageio
except ImportError:
    imageio = None

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "data-collection" else Path.cwd()

# Ensure rlab is in python path
RLAB_SRC = REPO_ROOT / "data-collection" / "agent" / "rlab" / "src"
if RLAB_SRC.exists() and str(RLAB_SRC) not in sys.path:
    sys.path.insert(0, str(RLAB_SRC))

from rlab.env import EnvConfig, make_eval_vec_env
from stable_baselines3 import PPO

RAW_EPISODES_DIR = REPO_ROOT / "data" / "raw" / "episodes"

print(f"Raw episodes will be written to: {RAW_EPISODES_DIR}")


Raw episodes will be written to: /Users/soheilchavoshi/Projects/mario-world/data/raw/episodes


In [30]:
@dataclass
class CollectionConfig:
    game: str = "SuperMarioBros-Nes-v0"
    level: str = "1-1"
    fps: int = 60
    uncap_fps: bool = True  # Uncap FPS by default to avoid timer sleep stutters and keep gameplay frame-accurate
    max_steps: int = 512
    action_set: str = "simple"
    policy_name: str = "ppo_pretrained"


CONFIG = CollectionConfig()
asdict(CONFIG) #convert to dictionary


{'game': 'SuperMarioBros-Nes-v0',
 'level': '1-1',
 'fps': 60,
 'uncap_fps': True,
 'max_steps': 4500,
 'action_set': 'simple',
 'policy_name': 'ppo_pretrained'}

In [31]:
import queue
import threading

def json_sanitize(obj):
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {str(k): json_sanitize(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [json_sanitize(v) for v in obj]
    elif isinstance(obj, (int, float, str, bool, type(None))):
        return obj
    else:
        return str(obj)


class EpisodeWriter:
    def __init__(self, episode_dir: Path, config: CollectionConfig):
        self.episode_dir = episode_dir
        self.tmp_dir = episode_dir.with_name(episode_dir.name + ".tmp")
        self.config = config
        self.q = queue.Queue()
        self.worker = threading.Thread(target=self._write_loop)

    def _write_loop(self):
        with imageio.get_writer(
            self.tmp_dir / "frames.mkv",
            fps=self.config.fps,
            codec="ffv1",
            macro_block_size=None,
        ) as video_writer, (self.tmp_dir / "steps.jsonl").open("w") as steps_file:
            while True:
                item = self.q.get()
                if item is None:
                    break
                frame, row = item
                video_writer.append_data(frame)
                steps_file.write(json.dumps(row, default=json_sanitize) + "\n")

    def __enter__(self):
        #Overwrite pre-existing episodes
        if self.episode_dir.exists():
            shutil.rmtree(self.episode_dir)
        if self.tmp_dir.exists():
            shutil.rmtree(self.tmp_dir)
        self.tmp_dir.mkdir(parents=True)

        #write meta file
        (self.tmp_dir / "meta.json").write_text(json.dumps(asdict(self.config), indent=2, default=json_sanitize) + "\n")
        self.worker.start()
        return self

    def write_step(self, t: int, frame: np.ndarray, action: int, reward: float, done: bool, info: dict[str, Any]):
        row = {
            "t": int(t),
            "action": int(action),
            "reward": float(reward),
            "done": bool(done),
            "info": json_sanitize(info),
        }
        self.q.put((np.asarray(frame, dtype=np.uint8), row))

    def __exit__(self, exc_type, exc, tb):
        self.q.put(None)
        self.worker.join()
        if exc_type is None:
            self.tmp_dir.rename(self.episode_dir)
        else:
            shutil.rmtree(self.tmp_dir, ignore_errors=True)
        return False


In [32]:
def make_env(config: CollectionConfig):
    rlab_config = EnvConfig(
        game=config.game,
        env_provider="supermariobrosnes-turbo",
        action_set=config.action_set,
        frame_skip=4,
        observation_size=84,
        obs_crop=(32, 0, 0, 0),
        max_pool_frames=False,
        max_episode_steps=4500,  # Keep full env max steps to prevent premature VecEnv auto-reset & frame-stack corruption
    )
    return make_eval_vec_env(rlab_config, n_envs=1, seed=random.randint(0, 1000000))


def load_policy(config: CollectionConfig):
    model_path = REPO_ROOT / "data-collection" / "agent" / "NES-SuperMarioBros_Level1-1_gray84-hudcrop-stack4-simple_ppo" / "model.zip"
    if not model_path.exists():
        model_path = REPO_ROOT / "training" / "agent" / "NES-SuperMarioBros_Level1-1_gray84-hudcrop-stack4-simple_ppo" / "model.zip"
    return PPO.load(model_path)


def unwrap_env(env):
    curr = env
    while True:
        if hasattr(curr, "venv"):
            curr = curr.venv
        elif hasattr(curr, "envs") and len(curr.envs) > 0:
            curr = curr.envs[0]
        elif hasattr(curr, "env"):
            curr = curr.env
        else:
            break
    return curr


def get_rgb_frame(env, obs) -> np.ndarray:
    if hasattr(env, "render"):
        frame = env.render(mode="rgb_array") if "mode" in getattr(env.render, "__code__", object()).co_varnames else env.render()
        if frame is not None:
            if isinstance(frame, list) and len(frame) > 0:
                frame = frame[0]
            return np.asarray(frame, dtype=np.uint8)
    return np.asarray(obs, dtype=np.uint8)


def policy_action(policy, obs, deterministic: bool = True):
    if hasattr(policy, "predict"):
        action, _ = policy.predict(obs, deterministic=deterministic)
        return action
    action = policy(obs)
    if isinstance(action, tuple):
        action = action[0]
    return action


def choose_action(env, policy, obs, config: CollectionConfig) -> int:
    action = policy_action(policy, obs, deterministic=True)
    return int(np.asarray(action).flat[0])


In [33]:
def run_episode(
    episode_id: int,
    config: CollectionConfig = CONFIG,
) -> dict[str, Any]:
    episode_dir = RAW_EPISODES_DIR / f"ep_{episode_id:06d}"
    env = make_env(config)
    policy = load_policy(config)

    total_reward = 0.0

    try:
        obs_res = env.reset()
        obs = obs_res[0] if isinstance(obs_res, tuple) else obs_res

        with EpisodeWriter(episode_dir, config) as writer:
            for t in range(config.max_steps):
                if not config.uncap_fps and config.fps > 0:
                    time.sleep(1.0 / config.fps)

                action = choose_action(env, policy, obs, config)
                step_res = env.step([action] if hasattr(env, "num_envs") else action)
                if len(step_res) == 4:
                    obs, reward, done, info_res = step_res
                    reward_val = float(reward[0]) if isinstance(reward, (list, np.ndarray)) else float(reward)
                    done_val = bool(done[0]) if isinstance(done, (list, np.ndarray)) else bool(done)
                    info_dict = info_res[0] if isinstance(info_res, list) and len(info_res) > 0 else info_res
                else:
                    obs, reward_val, terminated, truncated, info_dict = step_res
                    done_val = bool(terminated or truncated)

                total_reward += float(reward_val)

                frame = get_rgb_frame(env, obs)
                step_info = dict(info_dict or {})
                writer.write_step(t, frame, action, reward_val, done_val, step_info)

                if done_val:
                    break
    finally:
        env.close()

    return {"episode_id": episode_id, "steps": t + 1, "reward": total_reward, "dir": str(episode_dir)}


## Collection Plan

Collect raw episodes using the pre-trained PPO policy.


In [34]:
def make_collection_plan(num_episodes: int) -> list[CollectionConfig]:
    return [CONFIG] * num_episodes


def collect_dataset(num_episodes: int, dry_run: bool = True, overwrite: bool = True) -> list[dict[str, Any]]:
    plan = make_collection_plan(num_episodes)
    summaries = []

    for episode_id, config in enumerate(plan, start=1):
        episode_dir = RAW_EPISODES_DIR / f"ep_{episode_id:06d}"

        if episode_dir.exists() and not overwrite:
            summaries.append({"episode_id": episode_id, "status": "skipped_exists"})
            continue

        if dry_run:
            summaries.append({"episode_id": episode_id, "status": "planned"})
            continue

        summary = run_episode(episode_id, config)
        summary["status"] = "written"
        summaries.append(summary)

    return summaries


In [35]:
# Smoke test once make_env/load_policy are wired.
summary = run_episode(episode_id=1, config=CONFIG)
print("Smoke test episode summary:", summary)

# Verify recorded artifact
ep_dir = Path(summary["dir"])
assert ep_dir.exists(), f"Episode directory {ep_dir} does not exist!"

with imageio.get_reader(ep_dir / "frames.mkv") as video_reader:
    frame_count = sum(1 for _ in video_reader)

with (ep_dir / "steps.jsonl").open() as f:
    step_count = sum(1 for _ in f)

print(f"Recorded frame count: {frame_count}")
print(f"Recorded step count: {step_count}")
assert frame_count == step_count, f"Frame count ({frame_count}) does not match step count ({step_count})"

# Preview the 80-episode collection plan without writing files.
plan_preview = collect_dataset(num_episodes=5, dry_run=False)
print("Plan preview (first 5):", plan_preview[:5])
print("Total episodes planned:", len(plan_preview))

Smoke test episode summary: {'episode_id': 1, 'steps': 1036, 'reward': 322.9750105701387, 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/raw/episodes/ep_000001'}
Recorded frame count: 1036
Recorded step count: 1036
Plan preview (first 5): [{'episode_id': 1, 'steps': 1036, 'reward': 322.9750105701387, 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/raw/episodes/ep_000001', 'status': 'written'}, {'episode_id': 2, 'steps': 1036, 'reward': 322.9750105701387, 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/raw/episodes/ep_000002', 'status': 'written'}]
Total episodes planned: 2
